*Submitted to the Google - The Gemma 4 Developer Agent Paper Track.*

This notebook is generated from `paper/PAPER.md` in the linked repository; the repository is the primary artifact and contains the environment, task generator, policies, graders and committed result files.

# Bounded Context Without Context Rewrites

Roni Saguey

## Abstract

Long-horizon autonomous agents are limited by the economics of context rather than by model
capability. The standard remedy, periodic compaction, keeps the window small by rewriting it, and a
rewrite invalidates the shared prefix that a serving engine's prefix cache depends on, so the prompt
is billed again at the full rate. We describe a retention policy that bounds the working set without
rewriting the front: the system prompt and the first-turn brief are pinned in place, and the window
advances at the end. We evaluate three policies over sixty turns of a bug-fixing task on a 12B model,
measuring retention, compute cost, and prefix-cache reuse directly rather than inferring them. The
anchored policy holds its working set under a fixed budget and recovers every per-turn secret, sixty
of sixty against nine, while using 4.1 times less compute than the growing baseline at identical
coding accuracy. We also document four ways in which an evaluation of this kind can produce a
clean-looking table that measures nothing.

## 1. Introduction

An agent that must remain coherent across hundreds of turns accumulates history, and with it two
costs. The first is the memory footprint of the key and value cache; the second is the compute spent
re-processing the context on every turn. The dominant response in practice is compaction, which
summarises or discards the middle of the conversation in order to stay below the window.

Compaction reduces the size of the context but changes its content, and this matters for reasons
outside the agent. Serving engines maintain an automatic prefix cache, in which an input prefix
identical to the previous request reuses its key and value state, so a cached token costs a fraction
of a fresh one. A policy that rewrites the prefix therefore re-bills the entire context at the full
rate, and the more often it rewrites, the more it pays.

The constraint is concrete in the setting that motivates this work. The Gemma 4 Developer Agent
harness compacts the agent's history at 14,336 tokens, which is 2,048 tokens below the output ceiling
it also requests, and it acts on the prompt just sent rather than on the prompt plus the output, so
it is one call late. The scorer refuses any call beyond 32,768 tokens, and with a LoRA adapter
attached the KV cache on four L4 accelerators falls from roughly 46,000 to 7,600 tokens. Context
capacity, not reasoning, is the binding constraint, and compaction is the field's response to it.

We therefore ask a narrow and measurable question: can an agent keep a small context without
rewriting it? Our policy pins an immutable front, consisting of the system prompt and the first-turn
brief, and advances a bounded window at the end of the transcript. Nothing is summarised. The part of
each turn that carries unique information is kept, and the part that has been superseded is dropped.

The contributions of this paper are as follows. First, a context retention policy that bounds the
working set while leaving the prefix shared with the previous turn intact. Second, a per-turn
measurement of prefix-cache reuse that treats cost as a function of the retention policy rather than
a constant. Third, an evaluation that discriminates between policies, using per-turn grading and
retention probes that recency alone cannot satisfy. Fourth, an account of four failure modes by which
an evaluation of this kind silently measures nothing, drawn from defects we found in our own
benchmark.

## 2. Related Work

Work on KV-cache eviction reduces memory by scoring tokens and discarding those with low scores. H2O
[1], Scissorhands [2], TOVA [3], and SnapKV [4] all assume that the surviving tokens remain valid
after eviction, an assumption we do not share, for reasons we give in Section 6. A second line of
work repairs positional integrity directly: CacheBlend [5] recomputes part of the cache and re-encodes
positions, CacheFocus [6] re-positions after pruning, DSCache [7] stores pre-rotation keys, and
CacheGen [8] recomputes as a fallback. Our approach does none of this. We never rewrite the region on
which the cache depends, and it is precisely this restraint that allows reuse to survive.

Attention-sink work explains why pinning an immutable front is mechanically sensible. StreamingLLM
[9] retains a small set of sink tokens together with a sliding window, and the sink mechanism itself
has been characterised [10]. Sinks stabilise a stream, but they are not a channel for specific
information [11], which is why our probes test for a particular fact rather than for fluency.

The closest prior work is SinkTrack [20], which is nearest to our intent: it anchors a model to its
initial context and identifies context forgetting as a genuine failure mode of long generation. The
mechanisms differ in kind. SinkTrack is a model-level intervention that injects features into the
beginning-of-sequence representation, and it is evaluated on single-generation question answering. Our
policy operates at the level of the agent, deciding which tokens are retained and measuring the
serving-cost consequence over a long horizon. SinkTrack also provides a reason to expect a pinned
prefix to help, since a front that attracts attention is a front worth preserving.

Work on agent memory addresses what an agent should store. MemGPT [12] pages context in the manner of
an operating system, and a survey [13] taxonomises memory operations. Both concern storage rather
than the serving cost of rewriting what has been stored.

Position-dependent context use is well established. Lost in the Middle [14] and Found in the Middle
[15] show that models use information differently depending on where it occurs. Gemma-4 makes this
concrete: forty of its forty-eight layers use a 1024-token local-attention window, so a first-turn
fact is unavailable to those layers regardless of retention policy. The claims in this paper concern
the tokens the agent retained, and Section 6 states the architectural caveat that follows.

Finally, our own benchmark produced a result that we traced to a defect in the task rather than in
the policy, and we report the corrected version; Section 6 describes it. A position paper argues that
venues should establish a track for refutations and critiques [16]. The literature documents leakage
and reporting error in machine-learning research [17], and empirical-methodology work shows that
comparisons tend to favour the proposed method [18], which is the bias that two of our failure modes
produce. We follow established artifact-review expectations [19].

## 3. Method

### 3.1 Retention policies

Let the transcript be a sequence of turn blocks, where each turn appends one instruction and one
reply. The three policies we compare differ only in which blocks are retained when the prompt is
assembled.

The anchored policy retains the first-turn block, every instruction issued so far in verbatim form,
and the single most recent reply. Instructions are small, whereas a reply in this setting is a
complete file rewrite that the newest reply already supersedes, so retaining more than one reply is
redundant. If the budget is exceeded the oldest instructions are dropped first, so growth is one
instruction per turn and the front of the context is never rewritten.

The linear policy retains the first-turn block and as many recent instruction-reply pairs as fit the
ceiling, dropping the oldest block when the budget is exceeded. It is a fixed window with no internal
organisation, and it represents the naive baseline.

The prune policy is identical to the linear policy with the addition of a compaction every thirty
turns, which retains the anchor and the most recent turns. This is the status quo for long-running
agents.

All three policies share the same model, prompts, tools, turn budget, and first-turn anchor, so any
difference between them is attributable to the retention policy. The prune policy is anchored
deliberately, since otherwise it would differ from the linear policy in two ways at once and a
collapse could not be attributed to either.

### 3.2 Predictions

The claim under test is that stability, not size, is what a long-horizon agent requires. Two
separable predictions follow, because retention and cost are measured independently.

The first prediction concerns retention. A fact stated early and required at the end of the session
survives under the anchored policy and does not survive under the evicting policies, whose reach is a
fixed number of turns regardless of how large the window is. The second prediction concerns cost. The
anchored policy reuses a larger prefix because it never rewrites the front that the previous turn
already paid for, whereas the prune policy re-prefills at each compaction. A cost difference on its
own would not support the claim, so Section 5 reports the two measurements separately.

We also registered a prediction about the mechanism before running the experiment. The anchored
policy retains three things: the first-turn brief, the accumulated instructions, and the newest
reply. We credit the instruction archive, because it is the only home of each turn's code, since a
complete file rewrite supersedes every earlier reply. We therefore run an ablation that is the
anchored policy minus the archive and identical in every other respect. If the archive is the
mechanism, recall collapses to the few instructions that fit beside the brief while per-turn success
remains high. If instead the ablated policy retains all sixty, then the archive is not the mechanism
and our explanation is wrong, and we would report that outcome as such.

### 3.3 Task

The task is a Python module seeded with a number of faulty functions. The first turn carries the
complete bug report, which for each bug gives the exact wrong constant and the exact required
constant. Every later turn says only that a particular bug should be fixed. The list is never
repeated and no reminder is given, so the correctness of a later turn depends on the agent retaining
the first turn.

The task has four deliberate properties. Each bug family has the same shape, a one-line body with a
single wrong constant; an earlier version used a keyword default and a table lookup, both of which
invited the model to rewrite the shape and then fail with the brief only one turn back, which was a
defect in the task rather than a result about retention. The required constants are arbitrary and
appear nowhere else, so they cannot be re-derived. Grading is per turn, in which bug k is tested
immediately after turn k in a subprocess, so a fix that is applied and later clobbered by a rewrite is
distinguishable from one that was never applied. Finally, repetition is across task instances rather
than across sampled seeds. Decoding is greedy, so re-running a task is bit-identical and a seed would
fabricate variance; instead a task salt re-derives every constant into a new instance of the same
broken shape, which is a different task with the same retention probes, so runs remain comparable
while the work underneath them varies.

### 3.4 Retention probes

Retention is measured by two probes. The first is a set of accumulated codes. One fresh and
unguessable code is handed over in each turn and may not be written to the file until the final turn,
when it must be returned as an ordered list. The score runs from zero to n, so partial retention
appears as a gap in the prefix. The second is a release gate. On the final turn an additional
function must return a separate secret that was handed over at the midpoint of the session. This
secret is deliberately not one of the codes, so a policy can recite the list and still ship a broken
artifact; we verified that it is independent by checking that two simulated agents with identical bug
and code scores receive opposite gate verdicts.

The two policies diverge in growth rate by roughly an order of magnitude, which we measured live. The
linear policy adds approximately 1,248 tokens per turn, corresponding to a full instruction and reply
pair, while the anchored policy adds approximately 159, one instruction plus a single superseded
reply. Against a 12,000-token ceiling the linear policy reaches the limit at about turn nine and
evicts thereafter, holding the most recent four turn blocks, while the anchored policy holds every
instruction and finishes at approximately 9,934 tokens, under budget. This was computed against the
real tokenizer before the run and observed during it.

One probe failed to discriminate, and we report that as a negative result. We predicted that the
linear policy would fail the release gate, but it passed, as did the anchored policy. The reason is
that the gate's value is a function that the artifact already contains, so committing the secret to
that function when it is handed over carries it forward through every later rewrite. The gate
therefore measures artifact persistence rather than retention. The accumulated codes have no such
escape, because they are forbidden from the file until the final turn, which is why the codes probe
separates the arms in Section 5 while the gate does not.

## 4. Measurement

### 4.1 Prefix-cache reuse

On each turn we record the longest common token prefix between the current prompt and the previous
one, which is exactly what an automatic prefix cache would reuse. Cost is expressed in raw compute
units rather than in currency: a fresh token costs 1.0 and a reused token an assumed multiplier of
0.1, so that the cost of a turn is the number of miss tokens plus the number of hit tokens multiplied
by the hit multiplier, against a no-cache baseline in which every token costs 1.0. The multiplier is a
single named constant, so the assumption is visible and can be re-run with a different value.

We compare policies on total compute cost rather than on hit rate alone, because cost separates two
policies only if one both reuses its prefix and keeps the prompt small. A high hit rate on a prompt
that grows without bound still processes far more tokens than a short prompt does.

### 4.2 Metrics

The measurements fall into four groups. Retention is measured by per-turn success, final accuracy,
the number of fixes applied and then lost, code recall, and the gate verdict. Cost is measured by hit
rate, miss tokens, total compute units, the no-cache baseline, and the resulting saving ratio.
Context is measured by the first, last, and peak prompt sizes and the growth ratio. Latency is
measured by time to first token, wall time per turn, and decode throughput, with KV-cache peak and
total prefill as additional memory figures. All of these derive from data that the loop already
collects, so a single run yields the whole table rather than one run per question.

### 4.3 Fairness

The stopping rule is the turn count, not wall-clock time, and this is load-bearing. A policy that
keeps a smaller context is faster per turn by construction, so a time-based stop would allow it to
complete more turns and win on volume rather than on policy. Time is retained only as an emergency
break, so that an interrupted run still grades what it completed.

Because the emergency break can fire at different turn counts for different policies, the comparison
script reconciles to the common prefix, meaning the turns both policies completed, and re-sums cost,
context, and latency over that prefix. The measurements are per-turn and indexed, so the shared turns
are the same turns. The script refuses to compare when fewer than two turns are comparable. One
asymmetry is deliberate: the retention counts are not reconstructed, because the state of the
artifact at turn n cannot be recovered from a later snapshot, and they describe only the turns
reached.

## 5. Results

Both policies ran sixty comparable turns under an identical prompt and tool budget. Figure 1 shows
growth and reuse per turn. Table 1 reports cost, in which a reused token is priced at one tenth of a
fresh one and the baseline prices every token as a miss.

**Table 1. Compute cost over sixty turns.**

| policy | cache hit rate | miss tokens | cost units | no-cache cost | saving | prompt first to last | TTFT first to last (ms) |

|---|---|---|---|---|---|---|---|

| anchored | 81.3% | 82,854 | 118,817 | 442,488 | 3.72x | 2,057 to 9,934 | 1,786.2 to 5,592.0 |

| linear | 30.8% | 464,305 | 484,956 | 670,822 | 1.38x | 2,057 to 12,142 | 1,795.1 to 7,370.6 |

**Table 2. Retention.** Per-turn success is graded after each turn, code recall is the accumulated
list, and the release gate is the independent mid-session secret.

| policy | per-turn success | final state | applied then lost | code recall | ordered | release gate |

|---|---|---|---|---|---|---|

| anchored | 1.000 | 1.000 | 0 | 1.000 | yes | pass |

| linear | 1.000 | 1.000 | 0 | 0.150 | no | pass |

The anchored policy reuses a prefix for 81 percent of its prefill against 31 percent for the linear
policy, at 118,817 compute units against 484,956, a factor of 4.08. Coding accuracy is identical
between the two policies, at sixty of sixty, so the difference lies in what each retains rather than
in what the model can do. The separation is structural rather than incidental: a policy that evicts
from the front changes its first block as soon as eviction begins, whereas a policy that pins the
front and advances the window at the end keeps it, so reuse separates the two where context size
alone would not.

The third arm, the prune policy, behaves as the linear policy does. It scores sixty of sixty on the
task, recovers nine of sixty codes in unordered form, and passes the gate, with a cache hit rate of
0.310. The compaction at turn thirty is visible in the per-turn record as a collapse of the reused
prefix from 3,226 tokens to 2,057, which is the anchored prefix alone. Compaction therefore does not
recover the cost it is meant to recover; it pays a re-prefill at every boundary.

## 6. Failure Modes of the Evaluation

Each of the following produced, or would have produced, a clean-looking table that measured nothing.

The first failure mode is a shared ceiling. Two policies that are capped at the same limit converge
on whatever fits: with a single shared 12,000-token cap both policies reached approximately 9,900
tokens at nearly identical latency, 4,180 against 4,308 milliseconds, so the cap rather than the
policy set the working set. In the present design each budget is sized to the claim it supports.

The second failure mode is a re-derivable probe. A rule that the model re-applies on every turn can
be inferred from recent turns without the original, so both policies pass and the probe is vacuous.
Only a value stated once and never repeated discriminates between policies.

The third failure mode is a periodic task defect. Two of six bug families failed on every instance
from the second turn onward, which is one turn after the brief, so memory was not in question; their
shapes invited a rewrite. A turn-indexed task must be checked for periodic failure before its losses
are counted as retention results.

The fourth failure mode is an architectural confound. Gemma-4 uses local attention in forty of its
forty-eight layers with a 1024-token window, so beyond 1024 tokens a first-turn fact is unavailable
to those layers regardless of policy. Any claim about retaining an early fact must be stated against
that background. The KV-cache peak is not a result of the retention policy for the same reason, since
the model already bounds most of its own cache.

## 7. Limitations

The evaluation covers one model and one task family, and the task is synthetic by design so that
retention is measurable. The cost model uses an assumed hit multiplier rather than a measured billing
figure, and the sensitivity of the conclusion to that assumption is reported separately. Each policy
was run once per configuration, so the results carry no error bars, and repetition is across task
instances rather than seeds for the reason given in Section 3.3. The release gate probe is a single
value and establishes that one mid-session fact survived rather than a rate.

The reported run uses Gemma-4-12B in bfloat16, which loads on a single accelerator without a
quantization confound. The competition checkpoint is a 4-bit quantized model and would add a
quantization variable to a retention measurement. The policy is independent of the model, since it is
a decision about which blocks to retain and is implemented outside the model, so a different
checkpoint changes the numbers rather than the mechanism.

## 8. Conclusion

Keeping a context small and keeping it stable are different problems, and the usual solution to the
first destroys the second. Pinning an immutable front and rolling the window at the end bounds the
working set without invalidating the prefix that has already been paid for. We have made that claim
measurable rather than asserted, and we have documented how an evaluation of this kind can silently
fail to measure anything.

## Reproduction

```
python3 tools/run_compare.py benchmarks/results/incremental_gemma-4-12b_{linear,runtime}.json

```

The environment, task generator, policies, and graders are contained in the repository, and the
command above runs against committed result files. Policies are selected through the CCAI_ARM
environment variable, the retention gate is enabled with CCAI_GATE, and the compaction period is set
with CCAI_PRUNE_EVERY.

## References

[1] Zhang et al. H2O: Heavy-Hitter Oracle for Efficient Generative Inference of LLMs. NeurIPS 2023. arXiv:2306.14048
[2] Liu et al. Scissorhands: Exploiting the Persistence of Importance Hypothesis. NeurIPS 2023. arXiv:2305.17118
[3] Oren et al. Transformers are Multi-State RNNs. arXiv:2401.06104
[4] Li et al. SnapKV: LLM Knows What You Are Looking For Before Generation. NeurIPS 2024. arXiv:2404.14469
[5] Yao et al. CacheBlend: Fast LLM Serving for RAG with Cached Knowledge Fusion. EuroSys 2025. arXiv:2405.16444
[6] Lee, Park and Han. CacheFocus: Dynamic Cache Re-Positioning for Efficient RAG. arXiv:2502.11101
[7] Pang et al. Decouple and Cache: KV Cache Construction for Streaming Video. arXiv:2605.01858
[8] Liu et al. CacheGen: KV Cache Compression and Streaming for Fast LLM Serving. ACM SIGCOMM 2024. arXiv:2310.07240
[9] Xiao et al. Efficient Streaming Language Models with Attention Sinks. ICLR 2024. arXiv:2309.17453
[10] Gu et al. When Attention Sink Emerges in Language Models. arXiv:2410.10781
[11] Cao, Zhang and Tang. Separating Stream Stability from Long-Term Recall in Language Models. arXiv:2609.07282
[12] Packer et al. MemGPT: Towards LLMs as Operating Systems. arXiv:2310.08560
[13] Zhang et al. A Survey on the Memory Mechanism of LLM-based Agents. arXiv:2404.13501
[14] Liu et al. Lost in the Middle: How Language Models Use Long Contexts. TACL 2024. arXiv:2307.03172
[15] Hsieh, Chuang et al. Found in the Middle: Calibrating Positional Attention Bias Improves Long Context Utilization. arXiv:2406.16008
[16] Schaeffer, Kazdan, Denisov-Blanch, Miranda, Gerstgrasser et al. Position: Machine Learning Conferences Should Establish a Refutations and Critiques Track. NeurIPS 2025 Position Paper Track. arXiv:2506.19882
[17] Kapoor and Narayanan. Leakage and the reproducibility crisis in ML-based science. Patterns 2023.
[18] Herrmann et al. Why We Must Rethink Empirical Research in Machine Learning. ICML 2024.
[19] ACM. Artifact Review and Badging, v1.1. https://www.acm.org/publications/policies/artifact-review-and-badging-current
[20] Liu, Chen and Wang. SinkTrack: Attention Sink based Context Anchoring for Large Language Models. ICLR 2026. arXiv:2604.10027

## Figure 1

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1120 420" width="1120" height="420">
<rect width="1120" height="420" fill="#ffffff"/>
<line x1="74.0" y1="334.0" x2="544.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="74.0" y1="34.0" x2="74.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="74.0" y1="334.0" x2="544.0" y2="334.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="338.0" font-size="11" fill="#555" text-anchor="end">0</text>
<line x1="74.0" y1="259.0" x2="544.0" y2="259.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="263.0" font-size="11" fill="#555" text-anchor="end">5000</text>
<line x1="74.0" y1="184.0" x2="544.0" y2="184.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="188.0" font-size="11" fill="#555" text-anchor="end">10000</text>
<line x1="74.0" y1="109.0" x2="544.0" y2="109.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="113.0" font-size="11" fill="#555" text-anchor="end">15000</text>
<line x1="74.0" y1="34.0" x2="544.0" y2="34.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="38.0" font-size="11" fill="#555" text-anchor="end">20000</text>
<text x="74.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">0</text>
<text x="191.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">14</text>
<text x="309.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">29</text>
<text x="426.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">44</text>
<text x="544.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">59</text>
<text x="309.0" y="372.0" font-size="12" fill="#333" text-anchor="middle">turn</text>
<text x="24.0" y="184.0" font-size="12" fill="#333" text-anchor="middle" transform="rotate(-90 24.0 184.0)">prompt tokens (tokens)</text>
<text x="309.0" y="22.0" font-size="13" font-weight="600" fill="#111" text-anchor="middle">(a) context growth</text>
<polyline points="74.0,303.1 82.0,257.6 89.9,256.5 97.9,255.3 105.9,254.1 113.8,252.9 121.8,251.7 129.8,250.5 137.7,249.4 145.7,248.2 153.7,246.9 161.6,245.7 169.6,244.5 177.6,243.2 185.5,242.0 193.5,240.8 201.5,239.6 209.4,238.3 217.4,237.1 225.4,235.9 233.3,234.6 241.3,233.4 249.3,232.2 257.2,230.9 265.2,229.7 273.2,228.5 281.1,227.3 289.1,226.0 297.1,224.8 305.0,223.6 313.0,221.8 320.9,220.6 328.9,219.4 336.9,218.1 344.8,216.9 352.8,215.7 360.8,214.4 368.7,213.2 376.7,212.0 384.7,210.7 392.6,209.5 400.6,208.3 408.6,207.1 416.5,205.8 424.5,204.6 432.5,203.4 440.4,202.1 448.4,200.9 456.4,199.7 464.3,198.4 472.3,197.2 480.3,196.0 488.2,194.8 496.2,193.5 504.2,192.3 512.1,191.1 520.1,189.8 528.1,188.6 536.0,187.4 544.0,185.0" fill="none" stroke="#1b6ca8" stroke-width="2"/>
<polyline points="74.0,303.1 82.0,284.4 89.9,265.7 97.9,247.0 105.9,228.3 113.8,209.5 121.8,190.8 129.8,172.1 137.7,153.4 145.7,153.4 153.7,153.3 161.6,153.3 169.6,153.2 177.6,153.2 185.5,153.2 193.5,153.1 201.5,153.1 209.4,153.0 217.4,153.0 225.4,153.0 233.3,153.0 241.3,153.0 249.3,153.0 257.2,153.0 265.2,153.0 273.2,153.0 281.1,153.0 289.1,153.0 297.1,153.0 305.0,153.0 313.0,152.5 320.9,170.0 328.9,170.0 336.9,170.0 344.8,170.0 352.8,170.0 360.8,170.0 368.7,170.0 376.7,153.0 384.7,153.0 392.6,153.0 400.6,153.0 408.6,153.0 416.5,153.0 424.5,153.0 432.5,153.0 440.4,153.0 448.4,153.0 456.4,153.0 464.3,153.0 472.3,153.0 480.3,153.0 488.2,153.0 496.2,153.0 504.2,153.0 512.1,153.0 520.1,153.0 528.1,153.0 536.0,153.0 544.0,151.9" fill="none" stroke="#c0392b" stroke-width="2"/>
<polyline points="74.0,303.1 82.0,284.4 89.9,265.7 97.9,247.0 105.9,228.3 113.8,209.5 121.8,190.8 129.8,172.1 137.7,153.4 145.7,153.4 153.7,153.3 161.6,153.3 169.6,153.2 177.6,153.2 185.5,153.2 193.5,153.1 201.5,153.1 209.4,153.0 217.4,153.0 225.4,153.0 233.3,153.0 241.3,153.0 249.3,153.0 257.2,153.0 265.2,153.0 273.2,153.0 281.1,153.0 289.1,153.0 297.1,153.0 305.0,153.0 313.0,263.9 320.9,170.0 328.9,170.0 336.9,170.0 344.8,170.0 352.8,170.0 360.8,170.0 368.7,170.0 376.7,153.0 384.7,153.0 392.6,153.0 400.6,153.0 408.6,153.0 416.5,153.0 424.5,153.0 432.5,153.0 440.4,153.0 448.4,153.0 456.4,153.0 464.3,153.0 472.3,153.0 480.3,153.0 488.2,153.0 496.2,153.0 504.2,153.0 512.1,153.0 520.1,153.0 528.1,153.0 536.0,153.0 544.0,151.9" fill="none" stroke="#2e7d32" stroke-width="2"/>
<line x1="600.0" y1="334.0" x2="1070.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="600.0" y1="34.0" x2="600.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="600.0" y1="334.0" x2="1070.0" y2="334.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="338.0" font-size="11" fill="#555" text-anchor="end">0</text>
<line x1="600.0" y1="259.0" x2="1070.0" y2="259.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="263.0" font-size="11" fill="#555" text-anchor="end">25</text>
<line x1="600.0" y1="184.0" x2="1070.0" y2="184.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="188.0" font-size="11" fill="#555" text-anchor="end">50</text>
<line x1="600.0" y1="109.0" x2="1070.0" y2="109.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="113.0" font-size="11" fill="#555" text-anchor="end">75</text>
<line x1="600.0" y1="34.0" x2="1070.0" y2="34.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="38.0" font-size="11" fill="#555" text-anchor="end">100</text>
<text x="600.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">0</text>
<text x="717.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">14</text>
<text x="835.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">29</text>
<text x="952.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">44</text>
<text x="1070.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">59</text>
<text x="835.0" y="372.0" font-size="12" fill="#333" text-anchor="middle">turn</text>
<text x="550.0" y="184.0" font-size="12" fill="#333" text-anchor="middle" transform="rotate(-90 550.0 184.0)">reused (%)</text>
<text x="835.0" y="22.0" font-size="13" font-weight="600" fill="#111" text-anchor="middle">(b) prefix-cache reuse</text>
<polyline points="600.0,334.0 608.0,213.1 615.9,111.4 623.9,110.2 631.9,109.1 639.8,108.0 647.8,106.9 655.8,105.9 663.7,104.9 671.7,103.9 679.7,103.1 687.6,102.2 695.6,101.3 703.6,100.4 711.5,99.5 719.5,98.6 727.5,97.8 735.4,97.0 743.4,96.2 751.4,95.4 759.3,94.6 767.3,93.9 775.3,93.2 783.2,92.5 791.2,91.8 799.2,91.1 807.1,90.5 815.1,89.8 823.1,89.2 831.0,88.6 839.0,89.1 846.9,88.5 854.9,86.6 862.9,86.0 870.8,85.5 878.8,84.9 886.8,84.4 894.7,83.9 902.7,83.4 910.7,82.9 918.6,82.4 926.6,81.9 934.6,81.5 942.5,81.0 950.5,80.6 958.5,80.1 966.4,79.7 974.4,79.3 982.4,78.9 990.3,78.5 998.3,78.1 1006.3,77.7 1014.2,77.3 1022.2,76.9 1030.2,76.5 1038.1,76.2 1046.1,75.8 1054.1,75.4 1062.0,75.1 1070.0,76.8" fill="none" stroke="#1b6ca8" stroke-width="2"/>
<polyline points="600.0,334.0 608.0,147.6 615.9,116.5 623.9,98.7 631.9,87.3 639.8,79.3 647.8,73.4 655.8,68.8 663.7,65.2 671.7,253.6 679.7,253.7 687.6,253.7 695.6,253.7 703.6,253.7 711.5,253.7 719.5,253.8 727.5,253.8 735.4,253.8 743.4,253.8 751.4,253.8 759.3,253.8 767.3,253.8 775.3,253.8 783.2,253.8 791.2,253.8 799.2,253.8 807.1,253.8 815.1,253.8 823.1,253.8 831.0,253.8 839.0,254.0 846.9,277.7 854.9,277.5 862.9,277.5 870.8,277.5 878.8,277.5 886.8,277.5 894.7,277.6 902.7,283.0 910.7,253.8 918.6,253.8 926.6,253.8 934.6,253.8 942.5,253.8 950.5,253.8 958.5,253.8 966.4,253.8 974.4,253.8 982.4,253.8 990.3,253.8 998.3,253.8 1006.3,253.8 1014.2,253.8 1022.2,253.8 1030.2,253.8 1038.1,253.8 1046.1,253.8 1054.1,253.8 1062.0,253.8 1070.0,254.3" fill="none" stroke="#c0392b" stroke-width="2"/>
<polyline points="600.0,334.0 608.0,147.6 615.9,116.5 623.9,98.7 631.9,87.3 639.8,79.3 647.8,73.4 655.8,68.8 663.7,65.2 671.7,253.6 679.7,253.7 687.6,253.7 695.6,253.7 703.6,253.7 711.5,253.7 719.5,253.8 727.5,253.8 735.4,253.8 743.4,253.8 751.4,253.8 759.3,253.8 767.3,253.8 775.3,253.8 783.2,253.8 791.2,253.8 799.2,253.8 807.1,253.8 815.1,253.8 823.1,253.8 831.0,253.8 839.0,202.4 846.9,277.5 854.9,277.5 862.9,277.5 870.8,277.5 878.8,277.5 886.8,277.5 894.7,277.6 902.7,283.0 910.7,253.8 918.6,253.8 926.6,253.8 934.6,253.8 942.5,253.8 950.5,253.8 958.5,253.8 966.4,253.8 974.4,253.8 982.4,253.8 990.3,253.8 998.3,253.8 1006.3,253.8 1014.2,253.8 1022.2,253.8 1030.2,253.8 1038.1,253.8 1046.1,253.8 1054.1,253.8 1062.0,253.8 1070.0,254.3" fill="none" stroke="#2e7d32" stroke-width="2"/>
<line x1="920" y1="16" x2="942" y2="16" stroke="#1b6ca8" stroke-width="3"/>
<text x="948" y="20" font-size="12" fill="#333">runtime</text>
<line x1="920" y1="31" x2="942" y2="31" stroke="#c0392b" stroke-width="3"/>
<text x="948" y="35" font-size="12" fill="#333">linear</text>
<line x1="920" y1="46" x2="942" y2="46" stroke="#2e7d32" stroke-width="3"/>
<text x="948" y="50" font-size="12" fill="#333">prune</text>
</svg>

Growth (a) and prefix reuse (b), per turn, from the same per-turn records the tables are generated from. Source: `paper/figs/trajectory.svg`, regenerable with `tools/make_figures.py`.